# OpenDataCopilot - Exploration des Données

**Projet Master 2 Data Science**

Ce notebook explore les données téléchargées pour le projet OpenDataCopilot :
- **Santé publique** : Hospitalisations COVID-19, urgences, démographie médicale
- **Pollution** : GÉOD'AIR (national), Airparif (Île-de-France), OpenAQ (métadonnées)

---

## Sources de données pollution

| Source | Couverture | Polluants | Granularité | Rate Limit |
|--------|-----------|-----------|-------------|------------|
| **GÉOD'AIR** | France (10 villes) | NO2, PM2.5, PM10, O3 | Journalière | 15 req/h |
| **Airparif** | Île-de-France | NO2, PM2.5, PM10, O3 | Temps réel | Aucun |
| **OpenAQ** | Mondial | Divers | Temps réel | Limité |

> **Recommandation:** Utilisez GÉOD'AIR pour une couverture nationale fiable.

---

## Objectifs
1. Comprendre la structure et le contenu de chaque dataset
2. Identifier les colonnes clés pour le RAG
3. Détecter les problèmes de qualité des données
4. Visualiser les tendances et patterns
5. Préparer les données pour l'indexation vectorielle

---
# SECTION 1 - Configuration & Chargement des Données
---

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
import os

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# ═══════════════════════════════════════════════════════════
# Détection robuste du répertoire projet
# ═══════════════════════════════════════════════════════════
def find_project_root() -> Path:
    """Trouve la racine du projet en cherchant des fichiers marqueurs."""
    # Méthode 1: Chercher à partir du répertoire courant
    current = Path.cwd()
    
    # Fichiers qui indiquent la racine du projet
    markers = ['pyproject.toml', 'requirements.txt', '.git', 'README.md']
    
    # Remonter jusqu'à 5 niveaux
    for _ in range(5):
        if any((current / marker).exists() for marker in markers):
            return current
        current = current.parent
    
    # Méthode 2: Chemin absolu hardcodé en fallback
    fallback = Path("/Users/jerome/Documents/University/Master2/PROJET /Open_Data_Copilot")
    if fallback.exists():
        return fallback
    
    # Dernier recours
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_SANTE = PROJECT_ROOT / 'data' / 'raw' / 'sante'
DATA_POLLUTION = PROJECT_ROOT / 'data' / 'raw' / 'pollution'

print(f"📁 Racine projet: {PROJECT_ROOT}")
print(f"📁 Répertoire santé: {DATA_SANTE}")
print(f"📁 Répertoire pollution: {DATA_POLLUTION}")
print(f"   → Existe: {DATA_SANTE.exists()}")
print(f"   → Existe: {DATA_POLLUTION.exists()}")

📁 Racine projet: /Users/jerome/Documents/University/Master2/PROJET /Open_Data_Copilot
📁 Répertoire santé: /Users/jerome/Documents/University/Master2/PROJET /Open_Data_Copilot/data/raw/sante
📁 Répertoire pollution: /Users/jerome/Documents/University/Master2/PROJET /Open_Data_Copilot/data/raw/pollution
   → Existe: True
   → Existe: True


In [2]:
# Fonction de chargement sécurisé
def load_csv_safe(filepath: Path, **kwargs) -> pd.DataFrame | None:
    """Charge un CSV avec gestion d'erreurs."""
    try:
        df = pd.read_csv(filepath, **kwargs)
        print(f"✅ {filepath.name}: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
        return df
    except FileNotFoundError:
        print(f"❌ Fichier non trouvé: {filepath}")
        return None
    except Exception as e:
        print(f"❌ Erreur chargement {filepath.name}: {e}")
        return None

# Dictionnaire pour stocker tous les datasets
datasets = {}

In [3]:
# ═══════════════════════════════════════════════════════════
# Chargement des données SANTÉ
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📊 DONNÉES SANTÉ PUBLIQUE")
print("="*60 + "\n")

# Hospitalisations COVID-19
datasets['covid_hosp'] = load_csv_safe(
    DATA_SANTE / 'covid_hospitalisations.csv',
    sep=';',
    low_memory=False
)

# Urgences SurSaUD (incidences hospitalières)
datasets['urgences'] = load_csv_safe(
    DATA_SANTE / 'sursaud_urgences.csv',
    sep=';',
    low_memory=False
)

# Tests COVID (nouveau format depuis 2022)
datasets['covid_tests'] = load_csv_safe(
    DATA_SANTE / 'covid_tests_dep.csv',
    sep=';'
)

# Professionnels de santé - CORRECTION: utiliser sep=';'
datasets['medecins'] = load_csv_safe(
    DATA_SANTE / 'professionnels_sante_dep.csv',
    sep=';',
    encoding='utf-8-sig'  # Pour gérer le BOM UTF-8
)


📊 DONNÉES SANTÉ PUBLIQUE

✅ covid_hospitalisations.csv: 338,245 lignes × 10 colonnes
✅ sursaud_urgences.csv: 113,016 lignes × 6 colonnes


✅ covid_tests_dep.csv: 118,626 lignes × 9 colonnes
✅ professionnels_sante_dep.csv: 1,080 lignes × 9 colonnes


In [4]:
# ═══════════════════════════════════════════════════════════
# Chargement des données POLLUTION
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("🌍 DONNÉES POLLUTION")
print("="*60 + "\n")

# Airparif
datasets['airparif'] = load_csv_safe(
    DATA_POLLUTION / 'airparif_indices.csv'
)

# OpenAQ par ville
datasets['openaq_paris'] = load_csv_safe(
    DATA_POLLUTION / 'openaq_paris_latest.csv'
)
datasets['openaq_lyon'] = load_csv_safe(
    DATA_POLLUTION / 'openaq_lyon_latest.csv'
)
datasets['openaq_marseille'] = load_csv_safe(
    DATA_POLLUTION / 'openaq_marseille_latest.csv'
)


🌍 DONNÉES POLLUTION

✅ airparif_indices.csv: 761 lignes × 17 colonnes
✅ openaq_paris_latest.csv: 171 lignes × 7 colonnes
✅ openaq_lyon_latest.csv: 58 lignes × 7 colonnes
✅ openaq_marseille_latest.csv: 106 lignes × 7 colonnes


In [5]:
# Résumé des datasets chargés
print("\n" + "="*60)
print("📋 RÉSUMÉ DES DATASETS")
print("="*60)

summary_data = []
for name, df in datasets.items():
    if df is not None:
        memory_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
        summary_data.append({
            'Dataset': name,
            'Lignes': f"{df.shape[0]:,}",
            'Colonnes': df.shape[1],
            'Mémoire (MB)': f"{memory_mb:.2f}"
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)


📋 RÉSUMÉ DES DATASETS


,Dataset,Lignes,Colonnes,Mémoire (MB)
0,covid_hosp,"338,245",10,61.31
1,urgences,"113,016",6,17.04
2,covid_tests,"118,626",9,57.91
3,medecins,"1,080",9,0.47
4,airparif,761,17,0.38
5,openaq_paris,171,7,0.05
6,openaq_lyon,58,7,0.02
7,openaq_marseille,106,7,0.03


### Aperçu des premières lignes de chaque dataset

In [6]:
# Afficher les premières lignes de chaque dataset
for name, df in datasets.items():
    if df is not None:
        print(f"\n{'='*60}")
        print(f"📄 {name.upper()}")
        print(f"{'='*60}")
        display(df.head(3))
        print(f"\nColonnes: {list(df.columns)}")


📄 COVID_HOSP


,dep,sexe,jour,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
0,01,0,2020-03-18,2,0,NaN,NaN,NaN,1,0
1,01,1,2020-03-18,1,0,NaN,NaN,NaN,1,0
2,01,2,2020-03-18,1,0,NaN,NaN,NaN,0,0



Colonnes: ['dep', 'sexe', 'jour', 'hosp', 'rea', 'HospConv', 'SSR_USLD', 'autres', 'rad', 'dc']

📄 URGENCES


,dep,jour,incid_hosp,incid_rea,incid_dc,incid_rad
0,01,2020-03-19,1,0,0,0
1,01,2020-03-20,0,0,0,1
2,01,2020-03-21,3,0,0,0



Colonnes: ['dep', 'jour', 'incid_hosp', 'incid_rea', 'incid_dc', 'incid_rad']

📄 COVID_TESTS


...


Colonnes: ['location_id', 'location', 'city', 'country', 'parameter', 'last_value', 'last_updated']

📄 OPENAQ_LYON


,location_id,location,city,country,parameter,last_value,last_updated
0,2680,Lyon Périphérique,Lyon,FR,co,NaN,NaN
1,2680,Lyon Périphérique,Lyon,FR,no,NaN,NaN
2,2680,Lyon Périphérique,Lyon,FR,no2,NaN,NaN



Colonnes: ['location_id', 'location', 'city', 'country', 'parameter', 'last_value', 'last_updated']

📄 OPENAQ_MARSEILLE


,location_id,location,city,country,parameter,last_value,last_updated
0,3985,FR03014,Marseille,FR,no,NaN,NaN
1,3985,FR03014,Marseille,FR,no2,NaN,NaN
2,3985,FR03014,Marseille,FR,pm10,NaN,NaN



Colonnes: ['location_id', 'location', 'city', 'country', 'parameter', 'last_value', 'last_updated']


---
# SECTION 2 - Statistiques Descriptives
---

In [7]:
def analyze_dataset(name: str, df: pd.DataFrame) -> dict:
    """Analyse complète d'un dataset."""
    if df is None:
        return None
    
    print(f"\n{'═'*70}")
    print(f"📊 ANALYSE: {name.upper()}")
    print(f"{'═'*70}")
    
    # Dimensions
    print(f"\n📐 Dimensions: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    
    # Types de données
    print(f"\n📋 Types de données:")
    for dtype, count in df.dtypes.value_counts().items():
        print(f"   {dtype}: {count} colonnes")
    
    # Colonnes temporelles (détection automatique)
    date_cols = [c for c in df.columns if any(x in c.lower() for x in ['date', 'jour', 'time', 'semaine'])]
    if date_cols:
        print(f"\n📅 Colonnes temporelles détectées: {date_cols}")
        for col in date_cols[:2]:  # Limiter à 2
            try:
                dates = pd.to_datetime(df[col], errors='coerce')
                if dates.notna().any():
                    print(f"   {col}: {dates.min()} → {dates.max()}")
            except:
                pass
    
    # Colonnes géographiques
    geo_cols = [c for c in df.columns if any(x in c.lower() for x in ['dep', 'reg', 'ville', 'city', 'location'])]
    if geo_cols:
        print(f"\n🗺️ Colonnes géographiques: {geo_cols}")
        for col in geo_cols[:2]:
            n_unique = df[col].nunique()
            print(f"   {col}: {n_unique} valeurs uniques")
    
    # Statistiques numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Statistiques numériques:")
        display(df[numeric_cols].describe().round(2))
    
    return {
        'shape': df.shape,
        'date_cols': date_cols,
        'geo_cols': geo_cols,
        'numeric_cols': list(numeric_cols)
    }

In [8]:
# Analyser tous les datasets
analyses = {}
for name, df in datasets.items():
    if df is not None:
        analyses[name] = analyze_dataset(name, df)


══════════════════════════════════════════════════════════════════════
📊 ANALYSE: COVID_HOSP
══════════════════════════════════════════════════════════════════════

📐 Dimensions: 338,245 lignes × 10 colonnes

📋 Types de données:
   int64: 5 colonnes
   float64: 3 colonnes
   object: 2 colonnes

📅 Colonnes temporelles détectées: ['jour']


   jour: 2020-03-18 00:00:00 → 2023-03-31 00:00:00

🗺️ Colonnes géographiques: ['dep']
   dep: 102 valeurs uniques

📈 Statistiques numériques:


,sexe,hosp,rea,HospConv,SSR_USLD,autres,rad,dc
count,338245.00,338245.00,338245.00,228140.00,228140.00,228140.00,338245.00,338245.00
mean,1.00,113.80,13.36,61.50,35.98,2.99,2817.62,534.74
std,0.82,166.75,28.40,83.83,52.00,5.83,4173.61,730.47
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,23.00,1.00,15.00,6.00,0.00,500.00,105.00
50%,1.00,58.00,4.00,34.00,18.00,1.00,1404.00,278.00
75%,2.00,134.00,13.00,73.00,44.00,3.00,3338.00,653.00
max,2.00,3281.00,855.00,1115.00,565.00,170.00,48210.00,6463.00



══════════════════════════════════════════════════════════════════════
📊 ANALYSE: URGENCES
══════════════════════════════════════════════════════════════════════

📐 Dimensions: 113,016 lignes × 6 colonnes

📋 Types de données:
   int64: 4 colonnes
   object: 2 colonnes

📅 Colonnes temporelles détectées: ['jour']
   jour: 2020-03-19 00:00:00 → 2023-03-31 00:00:00

🗺️ Colonnes géographiques: ['dep']
   dep: 102 valeurs uniques

📈 Statistiques numériques:


,incid_hosp,incid_rea,incid_dc,incid_rad
count,113016.00,113016.00,113016.00,113016.00
mean,9.31,1.40,1.22,7.81
std,17.06,3.34,2.73,13.48
min,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00
50%,3.00,0.00,0.00,3.00
75%,11.00,1.00,1.00,9.00
max,404.00,96.00,76.00,239.00


...

,location_id,last_value,last_updated
count,171.00,0.00,0.00
mean,1261744.39,NaN,NaN
std,2147337.91,NaN,NaN
min,2674.00,NaN,NaN
25%,4072.50,NaN,NaN
50%,4123.00,NaN,NaN
75%,2292636.50,NaN,NaN
max,6216239.00,NaN,NaN



══════════════════════════════════════════════════════════════════════
📊 ANALYSE: OPENAQ_LYON
══════════════════════════════════════════════════════════════════════

📐 Dimensions: 58 lignes × 7 colonnes

📋 Types de données:
   object: 4 colonnes
   float64: 2 colonnes
   int64: 1 colonnes

📅 Colonnes temporelles détectées: ['last_updated']

🗺️ Colonnes géographiques: ['location_id', 'location', 'city']
   location_id: 16 valeurs uniques
   location: 16 valeurs uniques

📈 Statistiques numériques:


,location_id,last_value,last_updated
count,58.00,0.00,0.00
mean,1672569.78,NaN,NaN
std,2640515.98,NaN,NaN
min,2680.00,NaN,NaN
25%,3582.25,NaN,NaN
50%,3647.00,NaN,NaN
75%,4786675.50,NaN,NaN
max,6223217.00,NaN,NaN



══════════════════════════════════════════════════════════════════════
📊 ANALYSE: OPENAQ_MARSEILLE
══════════════════════════════════════════════════════════════════════

📐 Dimensions: 106 lignes × 7 colonnes

📋 Types de données:
   object: 4 colonnes
   float64: 2 colonnes
   int64: 1 colonnes

📅 Colonnes temporelles détectées: ['last_updated']

🗺️ Colonnes géographiques: ['location_id', 'location', 'city']
   location_id: 82 valeurs uniques
   location: 78 valeurs uniques

📈 Statistiques numériques:


,location_id,last_value,last_updated
count,106.00,0.00,0.00
mean,645789.54,NaN,NaN
std,952905.85,NaN,NaN
min,3985.00,NaN,NaN
25%,226367.50,NaN,NaN
50%,269244.50,NaN,NaN
75%,270542.75,NaN,NaN
max,5668064.00,NaN,NaN


---
# SECTION 3 - Visualisations Santé
---

## 3.1 Évolution des hospitalisations COVID-19

In [9]:
# Préparation des données COVID hospitalisations
if datasets['covid_hosp'] is not None:
    df_covid = datasets['covid_hosp'].copy()
    
    # Identifier la colonne de date
    date_col = None
    for col in ['jour', 'date', 'Date']:
        if col in df_covid.columns:
            date_col = col
            break
    
    if date_col:
        df_covid[date_col] = pd.to_datetime(df_covid[date_col], errors='coerce')
        print(f"Colonne de date utilisée: {date_col}")
        print(f"Plage: {df_covid[date_col].min()} → {df_covid[date_col].max()}")
    
    # Afficher les colonnes disponibles
    print(f"\nColonnes disponibles: {list(df_covid.columns)}")

Colonne de date utilisée: jour
Plage: 2020-03-18 00:00:00 → 2023-03-31 00:00:00

Colonnes disponibles: ['dep', 'sexe', 'jour', 'hosp', 'rea', 'HospConv', 'SSR_USLD', 'autres', 'rad', 'dc']


In [10]:
# Graphique 1: Évolution temporelle des hospitalisations France entière
if datasets['covid_hosp'] is not None and date_col:
    # Agréger par date (total France)
    hosp_col = None
    for col in ['hosp', 'hospitalisations', 'nb_hosp']:
        if col in df_covid.columns:
            hosp_col = col
            break
    
    if hosp_col:
        # Filtrer les données valides et agréger
        df_daily = df_covid.groupby(date_col)[hosp_col].sum().reset_index()
        df_daily = df_daily.dropna()
        
        fig = px.line(
            df_daily,
            x=date_col,
            y=hosp_col,
            title='📈 Évolution des hospitalisations COVID-19 en France',
            labels={date_col: 'Date', hosp_col: 'Nombre d\'hospitalisations'}
        )
        fig.update_layout(
            template='plotly_white',
            hovermode='x unified'
        )
        fig.show()
    else:
        print("Colonne d'hospitalisations non trouvée")

In [11]:
# Graphique 2: Top 10 départements les plus touchés
if datasets['covid_hosp'] is not None and hosp_col:
    dep_col = None
    for col in ['dep', 'departement', 'code_dep']:
        if col in df_covid.columns:
            dep_col = col
            break
    
    if dep_col:
        # Total par département
        df_by_dep = df_covid.groupby(dep_col)[hosp_col].sum().sort_values(ascending=False).head(10)
        
        fig = px.bar(
            x=df_by_dep.index.astype(str),
            y=df_by_dep.values,
            title='🏥 Top 10 départements - Hospitalisations COVID-19 cumulées',
            labels={'x': 'Département', 'y': 'Hospitalisations cumulées'},
            color=df_by_dep.values,
            color_continuous_scale='Reds'
        )
        fig.update_layout(template='plotly_white', showlegend=False)
        fig.show()

## 3.2 Passages aux urgences (SurSaUD)

In [12]:
# Analyse des urgences
if datasets['urgences'] is not None:
    df_urg = datasets['urgences'].copy()
    print(f"Colonnes urgences: {list(df_urg.columns)}")
    
    # Trouver la colonne de date
    date_col_urg = None
    for col in ['date_de_passage', 'date', 'jour']:
        if col in df_urg.columns:
            date_col_urg = col
            df_urg[col] = pd.to_datetime(df_urg[col], errors='coerce')
            break
    
    # Trouver colonnes de passages
    pass_cols = [c for c in df_urg.columns if 'pass' in c.lower() or 'nbre' in c.lower()]
    print(f"\nColonnes de passages: {pass_cols}")
    
    if pass_cols and date_col_urg:
        # Agréger par date
        numeric_cols = df_urg[pass_cols].select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            df_urg_daily = df_urg.groupby(date_col_urg)[numeric_cols].sum().reset_index()
            
            # Graphique
            fig = px.line(
                df_urg_daily,
                x=date_col_urg,
                y=numeric_cols[0],
                title='🚑 Évolution des passages aux urgences',
                labels={date_col_urg: 'Date', numeric_cols[0]: 'Nombre de passages'}
            )
            fig.update_layout(template='plotly_white')
            fig.show()

Colonnes urgences: ['dep', 'jour', 'incid_hosp', 'incid_rea', 'incid_dc', 'incid_rad']

Colonnes de passages: []


## 3.3 Démographie médicale

In [13]:
# Analyse de la démographie médicale
if datasets['medecins'] is not None:
    df_med = datasets['medecins'].copy()
    print(f"Colonnes médecins: {list(df_med.columns)}")
    display(df_med.head())
    
    # Chercher une colonne de région/département
    geo_col = None
    for col in ['departement', 'region', 'dep', 'territoire']:
        if col in df_med.columns:
            geo_col = col
            break
    
    # Chercher une colonne numérique pour les effectifs
    num_cols = df_med.select_dtypes(include=[np.number]).columns
    
    if geo_col and len(num_cols) > 0:
        # Agréger par géographie
        metric_col = num_cols[0]
        df_geo = df_med.groupby(geo_col)[metric_col].mean().sort_values(ascending=False).head(15)
        
        fig = px.bar(
            x=df_geo.values,
            y=df_geo.index.astype(str),
            orientation='h',
            title=f'👨‍⚕️ {metric_col} par {geo_col}',
            labels={'x': metric_col, 'y': geo_col},
            color=df_geo.values,
            color_continuous_scale='Blues'
        )
        fig.update_layout(template='plotly_white', showlegend=False, height=500)
        fig.show()

Colonnes médecins: ['annee', 'profession_sante', 'region', 'libelle_region', 'departement', 'libelle_departement', 'nombre_patients_medecin_traitant', 'taux_evolution_annuel', 'taux_evolution_annuel_integer']


,annee,profession_sante,region,libelle_region,departement,libelle_departement,nombre_patients_medecin_traitant,taux_evolution_annuel,taux_evolution_annuel_integer
0,2016,Médecins généralistes (hors médecins à experti...,1,Guadeloupe,999,Tout département,860,NC,NaN
1,2016,Médecins généralistes (hors médecins à experti...,2,Martinique,972,Martinique,891,NC,NaN
2,2016,Médecins généralistes (hors médecins à experti...,4,La Réunion,999,Tout département,775,NC,NaN
3,2016,Médecins généralistes (hors médecins à experti...,6,Mayotte,976,Mayotte,23,NC,NaN
4,2016,Médecins généralistes (hors médecins à experti...,6,Mayotte,999,Tout département,23,NC,NaN


---
# SECTION 4 - Visualisations Pollution
---

## 4.1 Indices Airparif (Île-de-France)

In [14]:
# Analyse des indices Airparif
if datasets['airparif'] is not None:
    df_air = datasets['airparif'].copy()
    print(f"Colonnes Airparif: {list(df_air.columns)}")
    display(df_air.head())
    
    # Statistiques
    print(f"\n📊 Statistiques Airparif:")
    display(df_air.describe())

Colonnes Airparif: ['FID', 'valeur', 'source', 'type_zone', 'code_zone', 'lib_zone', 'val_no2', 'val_so2', 'val_o3', 'val_pm10', 'val_pm25', 'couleur', 'x_lamb93', 'y_lamb93', 'date_ech', 'qualif', 'fetch_date']


,FID,valeur,source,type_zone,code_zone,lib_zone,val_no2,val_so2,val_o3,val_pm10,val_pm25,couleur,x_lamb93,y_lamb93,date_ech,qualif,fetch_date
0,1,3,Airparif,UU,851,Unité urbaine de Paris,2.00,1.00,3.00,2.00,0,#84BF75,652080,6862620,1578006000000,Bon,2026-02-01T23:15:14.431308
1,2,2,Airparif,UU,851,Unité urbaine de Paris,2.00,1.00,2.00,2.00,0,#84BF75,652080,6862620,1577919600000,Très bon,2026-02-01T23:15:14.431308
2,3,4,Airparif,UU,851,Unité urbaine de Paris,2.00,1.00,1.00,4.00,0,#84BF75,652080,6862620,1577833200000,Bon,2026-02-01T23:15:14.431308
3,4,8,Airparif,UU,851,Unité urbaine de Paris,3.00,1.00,1.00,8.00,0,#D82517,652080,6862620,1577746800000,Mauvais,2026-02-01T23:15:14.431308
4,5,7,Airparif,UU,851,Unité urbaine de Paris,3.00,1.00,1.00,7.00,0,#F29400,652080,6862620,1577660400000,Médiocre,2026-02-01T23:15:14.431308



📊 Statistiques Airparif:


,FID,valeur,code_zone,val_no2,val_so2,val_o3,val_pm10,val_pm25,x_lamb93,y_lamb93,date_ech
count,761.00,761.00,761.00,753.00,753.00,753.00,753.00,761.00,761.00,761.00,761.00
mean,627.40,3.81,851.00,2.16,1.00,3.31,3.09,0.00,652080.00,6862620.00,1579593855453.35
std,496.58,1.20,0.00,0.80,0.04,1.17,1.25,0.00,0.00,0.00,19234901676.57
min,1.00,2.00,851.00,1.00,0.00,1.00,1.00,0.00,652080.00,6862620.00,1546556400000.00
25%,191.00,3.00,851.00,2.00,1.00,3.00,2.00,0.00,652080.00,6862620.00,1562968800000.00
50%,381.00,3.00,851.00,2.00,1.00,3.00,3.00,0.00,652080.00,6862620.00,1579388400000.00
75%,1172.00,4.00,851.00,3.00,1.00,4.00,4.00,0.00,652080.00,6862620.00,1596319200000.00
max,1362.00,8.00,851.00,5.00,1.00,8.00,8.00,0.00,652080.00,6862620.00,1613430000000.00


In [15]:
# Distribution des indices Airparif
if datasets['airparif'] is not None:
    # Chercher une colonne d'indice
    idx_cols = [c for c in df_air.columns if 'indice' in c.lower() or 'qual' in c.lower() or 'lib' in c.lower()]
    num_cols = df_air.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(num_cols) >= 1:
        # Créer un histogramme pour chaque colonne numérique importante
        cols_to_plot = num_cols[:4]  # Limiter à 4 colonnes
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f'Distribution de {c}' for c in cols_to_plot[:4]]
        )
        
        colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
        
        for i, col in enumerate(cols_to_plot):
            row = i // 2 + 1
            col_idx = i % 2 + 1
            fig.add_trace(
                go.Histogram(x=df_air[col].dropna(), name=col, marker_color=colors[i]),
                row=row, col=col_idx
            )
        
        fig.update_layout(
            title='🌬️ Distribution des indices Airparif',
            template='plotly_white',
            height=500,
            showlegend=False
        )
        fig.show()

## 4.2 GÉOD'AIR - Données nationales (France entière)

**Source:** API GÉOD'AIR (https://www.geodair.fr)  
**Couverture:** 10 grandes villes françaises  
**Polluants:** NO2, PM2.5, PM10, O3  
**Granularité:** Moyennes journalières

> Note: Les données GÉOD'AIR complètent Airparif (Île-de-France) avec une couverture nationale.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Chargement des données GÉOD'AIR
# ═══════════════════════════════════════════════════════════
import glob

print("=" * 60)
print("🌬️ DONNÉES GÉOD'AIR (France nationale)")
print("=" * 60 + "\n")

# Chercher tous les fichiers GÉOD'AIR
geodair_files = list(DATA_POLLUTION.glob("geodair_*.csv"))

geodair_dfs = {}
if geodair_files:
    print(f"📁 {len(geodair_files)} fichiers GÉOD'AIR trouvés:\n")
    
    for filepath in sorted(geodair_files):
        try:
            df = pd.read_csv(filepath, sep=';', encoding='utf-8-sig')
            # Extraire le nom de la ville du fichier
            city = filepath.stem.split('_')[1].title()
            geodair_dfs[city] = df
            print(f"  ✅ {city}: {len(df):,} mesures ({filepath.name})")
        except Exception as e:
            print(f"  ❌ Erreur {filepath.name}: {e}")
    
    if geodair_dfs:
        # Combiner tous les DataFrames
        for city, df in geodair_dfs.items():
            df['ville'] = city
        df_geodair = pd.concat(geodair_dfs.values(), ignore_index=True)
        datasets['geodair'] = df_geodair
        
        print(f"\n📊 Total GÉOD'AIR: {len(df_geodair):,} mesures")
        print(f"   Villes: {list(geodair_dfs.keys())}")
else:
    print("⚠️ Aucun fichier GÉOD'AIR trouvé.")
    print("   Pour télécharger les données, exécutez:")
    print("   python -m data.pipelines.fetch_pollution --source geodair")
    print("\n   Note: L'API GÉOD'AIR a un rate limit de 15 requêtes/heure.")
    df_geodair = None

In [ ]:
# Analyse et visualisation GÉOD'AIR
if 'geodair' in datasets and datasets['geodair'] is not None:
    df_geo = datasets['geodair'].copy()
    
    print(f"Colonnes GÉOD'AIR: {list(df_geo.columns)}")
    display(df_geo.head())
    
    # Statistiques par polluant
    print("\n📊 Statistiques par polluant:")
    if 'Polluant' in df_geo.columns and 'valeur brute' in df_geo.columns:
        stats = df_geo.groupby('Polluant')['valeur brute'].describe()
        display(stats)
    
    # Graphique: Évolution temporelle par ville
    if 'Date' in df_geo.columns or 'Date de début' in df_geo.columns:
        date_col = 'Date' if 'Date' in df_geo.columns else 'Date de début'
        df_geo[date_col] = pd.to_datetime(df_geo[date_col], errors='coerce')
        
        if 'valeur brute' in df_geo.columns:
            # Box plot par ville
            fig = px.box(
                df_geo,
                x='ville',
                y='valeur brute',
                color='Polluant' if 'Polluant' in df_geo.columns else None,
                title='🏙️ Distribution des mesures GÉOD\'AIR par ville',
                labels={'valeur brute': 'Concentration', 'ville': 'Ville'}
            )
            fig.update_layout(template='plotly_white', height=500)
            fig.show()
else:
    print("⚠️ Données GÉOD'AIR non disponibles.")
    print("   Exécutez: python -m data.pipelines.fetch_pollution --source geodair")

## 4.3 Comparaison Paris vs Lyon vs Marseille (OpenAQ)

> **Note importante:** L'API OpenAQ v3 ne fournit plus de valeurs de mesures récentes pour les stations françaises.  
> Les métadonnées des stations restent disponibles mais les colonnes `last_value` et `last_updated` sont vides.  
> Pour des données de pollution fiables, utilisez **GÉOD'AIR** (couverture nationale) ou **Airparif** (Île-de-France).

In [16]:
# Combiner les données OpenAQ des 3 villes
openaq_dfs = []
for city, key in [('Paris', 'openaq_paris'), ('Lyon', 'openaq_lyon'), ('Marseille', 'openaq_marseille')]:
    if datasets.get(key) is not None:
        df_temp = datasets[key].copy()
        df_temp['city'] = city
        openaq_dfs.append(df_temp)
        print(f"{city}: {len(df_temp)} mesures")

if openaq_dfs:
    df_openaq = pd.concat(openaq_dfs, ignore_index=True)
    print(f"\n📊 Total OpenAQ: {len(df_openaq)} mesures")
    print(f"Colonnes: {list(df_openaq.columns)}")
    display(df_openaq.head())

Paris: 171 mesures
Lyon: 58 mesures
Marseille: 106 mesures

📊 Total OpenAQ: 335 mesures
Colonnes: ['location_id', 'location', 'city', 'country', 'parameter', 'last_value', 'last_updated']


,location_id,location,city,country,parameter,last_value,last_updated
0,2674,GARCHES,Paris,FR,o3,NaN,NaN
1,2675,CACHAN,Paris,FR,o3,NaN,NaN
2,2681,Place de l'Opéra,Paris,FR,no,NaN,NaN
3,2681,Place de l'Opéra,Paris,FR,no2,NaN,NaN
4,2681,Place de l'Opéra,Paris,FR,pm10,NaN,NaN


In [17]:
# Graphique comparatif par ville et paramètre
if openaq_dfs:
    # Chercher la colonne de paramètre (polluant) et valeur
    param_col = None
    value_col = None
    
    for col in ['parameter', 'param', 'polluant']:
        if col in df_openaq.columns:
            param_col = col
            break
    
    for col in ['last_value', 'value', 'valeur', 'moyenne']:
        if col in df_openaq.columns:
            value_col = col
            break
    
    if param_col and value_col:
        # Convertir les valeurs en numérique
        df_openaq[value_col] = pd.to_numeric(df_openaq[value_col], errors='coerce')
        
        # Vérifier si on a des valeurs non-nulles
        valid_data = df_openaq.dropna(subset=[value_col])
        
        if len(valid_data) > 0:
            # Box plot par polluant et ville
            fig = px.box(
                valid_data,
                x=param_col,
                y=value_col,
                color='city',
                title='🏙️ Comparaison des niveaux de pollution - Paris vs Lyon vs Marseille',
                labels={param_col: 'Polluant', value_col: 'Concentration', 'city': 'Ville'}
            )
            fig.update_layout(template='plotly_white', height=500)
            fig.show()
        else:
            print("⚠️ OpenAQ: Aucune valeur de mesure disponible")
            print("   L'API OpenAQ ne fournit plus de données récentes pour la France.")
            print("   Les données Airparif restent disponibles pour l'Île-de-France.")
            
            # Afficher la distribution des types de polluants mesurés
            if param_col in df_openaq.columns:
                print(f"\n📊 Polluants monitorés (métadonnées):")
                print(df_openaq[param_col].value_counts())
    else:
        print(f"Colonnes trouvées - param: {param_col}, value: {value_col}")

⚠️ OpenAQ: Aucune valeur de mesure disponible
   L'API OpenAQ ne fournit plus de données récentes pour la France.
   Les données Airparif restent disponibles pour l'Île-de-France.

📊 Polluants monitorés (métadonnées):
parameter
pm25                114
no2                  55
no                   45
pm10                 37
o3                   25
so2                  10
pm1                  10
relativehumidity     10
temperature          10
um003                10
co                    9
Name: count, dtype: int64


In [18]:
# Graphique en barres groupées (si données disponibles)
if openaq_dfs and param_col and value_col:
    # Vérifier si on a des valeurs valides
    valid_data = df_openaq.dropna(subset=[value_col])
    
    if len(valid_data) > 0:
        # Moyenne par ville et polluant
        df_summary = valid_data.groupby(['city', param_col])[value_col].mean().reset_index()
        
        fig = px.bar(
            df_summary,
            x=param_col,
            y=value_col,
            color='city',
            barmode='group',
            title='📊 Concentration moyenne par polluant et ville',
            labels={param_col: 'Polluant', value_col: 'Concentration moyenne', 'city': 'Ville'}
        )
        fig.update_layout(template='plotly_white')
        fig.show()
    else:
        # Alternative: afficher la couverture des stations par ville
        coverage = df_openaq.groupby(['city', param_col]).size().reset_index(name='stations')
        
        fig = px.bar(
            coverage,
            x=param_col,
            y='stations',
            color='city',
            barmode='group',
            title='📊 Couverture des stations OpenAQ par polluant et ville',
            labels={param_col: 'Polluant', 'stations': 'Nombre de capteurs', 'city': 'Ville'}
        )
        fig.update_layout(template='plotly_white')
        fig.show()
        print("\n💡 Note: Les données de mesures ne sont pas disponibles, mais les métadonnées des stations sont conservées.")


💡 Note: Les données de mesures ne sont pas disponibles, mais les métadonnées des stations sont conservées.


---
# SECTION 5 - Qualité des Données
---

In [19]:
def analyze_data_quality(name: str, df: pd.DataFrame):
    """Analyse la qualité d'un dataset."""
    if df is None:
        return None
    
    print(f"\n{'═'*60}")
    print(f"🔍 QUALITÉ DES DONNÉES: {name.upper()}")
    print(f"{'═'*60}")
    
    # Valeurs manquantes
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    missing_df = pd.DataFrame({
        'Colonne': missing.index,
        'Manquantes': missing.values,
        'Pourcentage': missing_pct.values
    })
    missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)
    
    if len(missing_df) > 0:
        print(f"\n⚠️ Colonnes avec valeurs manquantes:")
        display(missing_df.head(10))
    else:
        print(f"\n✅ Aucune valeur manquante!")
    
    # Doublons
    n_duplicates = df.duplicated().sum()
    print(f"\n📋 Doublons: {n_duplicates:,} ({n_duplicates/len(df)*100:.2f}%)")
    
    # Valeurs négatives (pour colonnes numériques)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    negatives = {}
    for col in numeric_cols:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            negatives[col] = n_neg
    
    if negatives:
        print(f"\n⚠️ Valeurs négatives détectées:")
        for col, count in negatives.items():
            print(f"   {col}: {count:,}")
    
    return {
        'missing': missing_df,
        'duplicates': n_duplicates,
        'negatives': negatives
    }

In [20]:
# Analyser la qualité de tous les datasets
quality_reports = {}
for name, df in datasets.items():
    if df is not None:
        quality_reports[name] = analyze_data_quality(name, df)


════════════════════════════════════════════════════════════
🔍 QUALITÉ DES DONNÉES: COVID_HOSP
════════════════════════════════════════════════════════════

⚠️ Colonnes avec valeurs manquantes:


,Colonne,Manquantes,Pourcentage
5,HospConv,110105,32.55
6,SSR_USLD,110105,32.55
7,autres,110105,32.55



📋 Doublons: 0 (0.00%)

════════════════════════════════════════════════════════════
🔍 QUALITÉ DES DONNÉES: URGENCES
════════════════════════════════════════════════════════════

✅ Aucune valeur manquante!

📋 Doublons: 0 (0.00%)

════════════════════════════════════════════════════════════
🔍 QUALITÉ DES DONNÉES: COVID_TESTS
════════════════════════════════════════════════════════════

⚠️ Colonnes avec valeurs manquantes:


,Colonne,Manquantes,Pourcentage
6,Tp,306,0.26



📋 Doublons: 0 (0.00%)

════════════════════════════════════════════════════════════
🔍 QUALITÉ DES DONNÉES: MEDECINS
════════════════════════════════════════════════════════════

⚠️ Colonnes avec valeurs manquantes:


...


📋 Doublons: 0 (0.00%)

════════════════════════════════════════════════════════════
🔍 QUALITÉ DES DONNÉES: OPENAQ_LYON
════════════════════════════════════════════════════════════

⚠️ Colonnes avec valeurs manquantes:


,Colonne,Manquantes,Pourcentage
5,last_value,58,100.00
6,last_updated,58,100.00



📋 Doublons: 0 (0.00%)

════════════════════════════════════════════════════════════
🔍 QUALITÉ DES DONNÉES: OPENAQ_MARSEILLE
════════════════════════════════════════════════════════════

⚠️ Colonnes avec valeurs manquantes:


,Colonne,Manquantes,Pourcentage
5,last_value,106,100.00
6,last_updated,106,100.00



📋 Doublons: 0 (0.00%)


In [21]:
# Heatmap des valeurs manquantes
print("\n" + "="*60)
print("🗺️ HEATMAP DES VALEURS MANQUANTES")
print("="*60)

# Créer un résumé
missing_summary = []
for name, df in datasets.items():
    if df is not None:
        pct_missing = df.isnull().mean().mean() * 100
        missing_summary.append({
            'Dataset': name,
            'Taux moyen manquant (%)': round(pct_missing, 2)
        })

if missing_summary:
    ms_df = pd.DataFrame(missing_summary)
    
    fig = px.bar(
        ms_df,
        x='Dataset',
        y='Taux moyen manquant (%)',
        title='📊 Taux de valeurs manquantes par dataset',
        color='Taux moyen manquant (%)',
        color_continuous_scale='RdYlGn_r'
    )
    fig.update_layout(template='plotly_white')
    fig.show()


🗺️ HEATMAP DES VALEURS MANQUANTES


---
# SECTION 6 - Colonnes Clés pour le RAG
---

In [ ]:
# Documentation des colonnes clés
print("═" * 70)
print("📋 COLONNES CLÉS POUR LE RAG")
print("═" * 70)

rag_columns = {
    'SANTÉ': {
        'covid_hosp': {
            'temporelles': ['jour'],
            'géographiques': ['dep', 'sexe'],
            'métriques': ['hosp', 'rea', 'rad', 'dc'],
            'description': 'Hospitalisations COVID-19 par département et sexe'
        },
        'urgences': {
            'temporelles': ['date_de_passage'],
            'géographiques': ['dep'],
            'métriques': ['nbre_pass_corona', 'nbre_pass_tot', 'nbre_hospit_corona'],
            'description': 'Passages aux urgences pour suspicion COVID'
        },
        'medecins': {
            'temporelles': ['annee'],
            'géographiques': ['region', 'departement'],
            'métriques': ['patientele_moyenne', 'effectif'],
            'description': 'Démographie des médecins généralistes'
        }
    },
    'POLLUTION': {
        'geodair': {
            'temporelles': ['Date de début', 'Date de fin'],
            'géographiques': ['ville', 'Organisme', 'Station'],
            'métriques': ['valeur brute', 'valeur nette', 'taux de saisie'],
            'description': 'Mesures nationales qualité air (10 villes, RECOMMANDÉ)'
        },
        'airparif': {
            'temporelles': ['fetch_date'],
            'géographiques': ['commune', 'station'],
            'métriques': ['indice', 'no2', 'pm25', 'pm10', 'o3'],
            'description': 'Indices qualité air Île-de-France'
        },
        'openaq': {
            'temporelles': ['last_updated'],
            'géographiques': ['city', 'location'],
            'métriques': ['last_value', 'parameter'],
            'description': 'Métadonnées stations (valeurs non disponibles pour la France)'
        }
    }
}

for domain, datasets_info in rag_columns.items():
    print(f"\n{'─'*60}")
    print(f"📂 {domain}")
    print(f"{'─'*60}")
    
    for ds_name, info in datasets_info.items():
        print(f"\n  📄 {ds_name}")
        print(f"     Description: {info['description']}")
        print(f"     📅 Temporelles: {info['temporelles']}")
        print(f"     🗺️  Géographiques: {info['géographiques']}")
        print(f"     📈 Métriques: {info['métriques']}")

In [23]:
# Tableau récapitulatif des colonnes
print("\n" + "═" * 70)
print("📊 TABLEAU RÉCAPITULATIF")
print("═" * 70)

recap_data = []
for name, df in datasets.items():
    if df is not None:
        recap_data.append({
            'Dataset': name,
            'Lignes': f"{len(df):,}",
            'Colonnes': len(df.columns),
            'Col. numériques': len(df.select_dtypes(include=[np.number]).columns),
            'Col. texte': len(df.select_dtypes(include=['object']).columns),
            'Valeurs manquantes': f"{df.isnull().sum().sum():,}"
        })

recap_df = pd.DataFrame(recap_data)
display(recap_df)


══════════════════════════════════════════════════════════════════════
📊 TABLEAU RÉCAPITULATIF
══════════════════════════════════════════════════════════════════════


,Dataset,Lignes,Colonnes,Col. numériques,Col. texte,Valeurs manquantes
0,covid_hosp,"338,245",10,8,2,"330,315"
1,urgences,"113,016",6,4,2,0
2,covid_tests,"118,626",9,1,8,306
3,medecins,"1,080",9,4,5,120
4,airparif,761,17,11,6,32
5,openaq_paris,171,7,3,4,342
6,openaq_lyon,58,7,3,4,116
7,openaq_marseille,106,7,3,4,212


---
# SECTION 7 - Insights & Recommandations
---

## 7.1 Patterns temporels identifiés

In [ ]:
print("═" * 70)
print("💡 INSIGHTS & DÉCOUVERTES")
print("═" * 70)

insights = [
    "📊 DONNÉES SANTÉ:",
    "   • Les hospitalisations COVID-19 montrent des vagues épidémiques distinctes",
    "   • Les départements urbains (75, 13, 69) sont les plus touchés",
    "   • Données disponibles depuis 2020, granularité quotidienne",
    "",
    "🌍 DONNÉES POLLUTION:",
    "   • GÉOD'AIR: couverture nationale, 10 grandes villes (RECOMMANDÉ)",
    "   • Airparif: couverture Île-de-France avec ~761 points de mesure",
    "   • OpenAQ: métadonnées uniquement (API v3 ne fournit plus de valeurs pour la France)",
    "   • Polluants principaux: NO2, PM2.5, PM10, O3",
    "",
    "⚠️ LIMITATIONS API:",
    "   • GÉOD'AIR: Rate limit de 15 requêtes/heure",
    "   • OpenAQ: Aucune valeur de mesure disponible pour la France",
    "",
    "🔗 CORRÉLATIONS POTENTIELLES:",
    "   • Pics de pollution → augmentation passages urgences respiratoires",
    "   • Saisonnalité: pollution hiver (chauffage) vs été (ozone)",
    "   • Géographie: zones urbaines = plus de pollution + plus d'hospitalisations"
]

for line in insights:
    print(line)

## 7.2 Recommandations pour le RAG

In [25]:
print("\n" + "═" * 70)
print("📝 RECOMMANDATIONS POUR LE RAG")
print("═" * 70)

recommendations = [
    "",
    "1️⃣  CHUNKING STRATEGY:",
    "    • Découper par département + période (semaine/mois)",
    "    • Inclure métadonnées: source, date_maj, unités",
    "    • Taille recommandée: 500-1000 tokens par chunk",
    "",
    "2️⃣  METADATA ENRICHMENT:",
    "    • Ajouter libellés des départements (ex: 75 → Paris)",
    "    • Normaliser les dates au format ISO",
    "    • Taguer le domaine: 'santé' ou 'pollution'",
    "",
    "3️⃣  QUESTIONS TYPES À SUPPORTER:",
    "    • 'Hospitalisations COVID à Paris en janvier 2024?'",
    "    • 'Niveau de NO2 à Lyon aujourd'hui?'",
    "    • 'Corrélation pollution-urgences à Marseille?'",
    "    • 'Densité médicale dans l'Hérault?'",
    "",
    "4️⃣  LIMITATIONS À DOCUMENTER:",
    "    • OpenAQ: données temps réel uniquement (pas d'historique)",
    "    • Médecins: données annuelles (pas temps réel)",
    "    • Airparif: couverture Île-de-France uniquement"
]

for line in recommendations:
    print(line)


══════════════════════════════════════════════════════════════════════
📝 RECOMMANDATIONS POUR LE RAG
══════════════════════════════════════════════════════════════════════

1️⃣  CHUNKING STRATEGY:
    • Découper par département + période (semaine/mois)
    • Inclure métadonnées: source, date_maj, unités
    • Taille recommandée: 500-1000 tokens par chunk

2️⃣  METADATA ENRICHMENT:
    • Ajouter libellés des départements (ex: 75 → Paris)
    • Normaliser les dates au format ISO
    • Taguer le domaine: 'santé' ou 'pollution'

3️⃣  QUESTIONS TYPES À SUPPORTER:
    • 'Hospitalisations COVID à Paris en janvier 2024?'
    • 'Niveau de NO2 à Lyon aujourd'hui?'
    • 'Corrélation pollution-urgences à Marseille?'
    • 'Densité médicale dans l'Hérault?'

4️⃣  LIMITATIONS À DOCUMENTER:
    • OpenAQ: données temps réel uniquement (pas d'historique)
    • Médecins: données annuelles (pas temps réel)
    • Airparif: couverture Île-de-France uniquement


## 7.3 Prochaines étapes

In [26]:
print("\n" + "═" * 70)
print("🚀 PROCHAINES ÉTAPES")
print("═" * 70)

next_steps = [
    "",
    "□ 1. Nettoyer et préprocesser les données",
    "     - Gérer les valeurs manquantes",
    "     - Normaliser les formats de dates",
    "     - Enrichir avec libellés départements/régions",
    "",
    "□ 2. Créer les documents pour le RAG",
    "     - Transformer les DataFrames en texte structuré",
    "     - Appliquer la stratégie de chunking",
    "     - Ajouter les métadonnées",
    "",
    "□ 3. Implémenter la baseline (sans RAG)",
    "     - Tester GPT-3.5-turbo sur les questions types",
    "     - Mesurer les hallucinations",
    "     - Établir les métriques de référence",
    "",
    "□ 4. Implémenter RAG Basic (FAISS)",
    "     - Générer les embeddings",
    "     - Créer l'index FAISS",
    "     - Tester la récupération"
]

for line in next_steps:
    print(line)


══════════════════════════════════════════════════════════════════════
🚀 PROCHAINES ÉTAPES
══════════════════════════════════════════════════════════════════════

□ 1. Nettoyer et préprocesser les données
     - Gérer les valeurs manquantes
     - Normaliser les formats de dates
     - Enrichir avec libellés départements/régions

□ 2. Créer les documents pour le RAG
     - Transformer les DataFrames en texte structuré
     - Appliquer la stratégie de chunking
     - Ajouter les métadonnées

□ 3. Implémenter la baseline (sans RAG)
     - Tester GPT-3.5-turbo sur les questions types
     - Mesurer les hallucinations
     - Établir les métriques de référence

□ 4. Implémenter RAG Basic (FAISS)
     - Générer les embeddings
     - Créer l'index FAISS
     - Tester la récupération


---
## Fin du notebook d'exploration

**Auteur:** Jérôme - Master 2 Data Science  
**Date:** Février 2025  
**Projet:** OpenDataCopilot

---